# 06 — Is it any good? accuracy and a baseline

*Module 00 · Getting Started — notebook 6 of 11*

We have been saying "the fraction it gets right" without quite pinning it down. This notebook names that number — **accuracy** — measures it properly, gives it a yardstick, and shows the one situation where it lies to you.

**Prerequisites:** 03 (class balance), 04 (the split), 05 (the nearest-centroid classifier).

**What you'll be able to do**
- Define **accuracy** and compute it, by hand and with scikit-learn.
- Compare a model to a **majority baseline**, and say why beating it matters.
- Explain the **accuracy paradox**: why a high accuracy can hide a useless model.
- Name the **positive class** and why it matters when one class is the one you care about.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestCentroid

from ml_course import colors, datasets, viz

np.random.seed(0)
viz.use_course_style()

X, y = datasets.penguins_xy()
FEATURES = datasets.PENGUINS_FEATURES
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

# The classifier from notebook 05, as its scikit-learn twin.
clf = NearestCentroid().fit(X_train, y_train)

## Where we are

Notebook 05 built a classifier and reported the fraction of held-out penguins it got right. We leaned on that number without defining it or asking how much to trust it. Now we do both — because a single accuracy figure, taken at face value, is one of the easiest ways to fool yourself in machine learning.

## Accuracy, defined

The number we have been eyeballing has a name: **accuracy**. It is the fraction of examples the model labels correctly:

$$ \text{accuracy} = \frac{\text{number correct}}{\text{total number of examples}}. $$

Nothing more. Let's compute it for our classifier on the held-out test set, by hand and with scikit-learn.

In [ ]:
pred = clf.predict(X_test)
n_correct = int((pred == y_test.to_numpy()).sum())
n_total = len(y_test)

print(f"accuracy by hand : {n_correct / n_total:.3f}  ({n_correct} correct out of {n_total})")
print(f"accuracy_score   : {accuracy_score(y_test, pred):.3f}")

### Read the output

Accuracy is the count of correct answers over the total — here **1.000**, every one of the 69 test penguins. The by-hand value and scikit-learn's `accuracy_score` agree, as they must: accuracy is not subtle. (Recall notebook 05's caveat — this perfect score reflects how cleanly these two species separate, not that classification is always this kind.)

## A score needs a yardstick: the baseline

A number on its own means little. Is 90% good? It depends on what an effortless rule would score. The simplest reference is a model that ignores the features entirely and **always predicts the most common class**. Any classifier worth its name has to beat that. scikit-learn provides it as a `DummyClassifier`.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, baseline.predict(X_test))
clf_acc = accuracy_score(y_test, pred)

print(f"majority baseline accuracy : {baseline_acc:.3f}")
print(f"nearest-centroid accuracy  : {clf_acc:.3f}")

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.bar(
    ["majority\nbaseline", "nearest\ncentroid"],
    [baseline_acc, clf_acc],
    color=[colors.COLORS["muted"], colors.COLORS["model"]],
    edgecolor=colors.COLORS["text"],
    linewidth=0.6,
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("accuracy on the test set")
for i, v in enumerate([baseline_acc, clf_acc]):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", color=colors.COLORS["text"])
ax.grid(False)
plt.show()

### Read the figure

The majority baseline scores 0.55 — it always answers "Adélie", the more common species in the training set, and is right whenever the true answer happens to be Adélie. (This is no accident: a majority baseline's accuracy is exactly the majority class's share of the test set — 38 of 69 here.) Our classifier reaches 1.00, far above it. That gap is the real signal: it tells us the model learned something from the measurements rather than riding the class split. A model that only matched the baseline would have learned nothing useful, however respectable its raw accuracy looked.

## Where accuracy misleads

Accuracy weighs every penguin equally. So when one class greatly outnumbers the other, accuracy mostly reports how well we do on the majority — and a model can score high while missing the minority entirely. Let's make that happen on purpose, with a deliberately imbalanced version of our test set.

In [ ]:
# A what-if: keep every Adélie test penguin, but only ~5% of the Gentoo ones.
test = X_test.copy()
test["species"] = y_test.to_numpy()
adelie = test[test["species"] == "Adelie"]
gentoo = test[test["species"] == "Gentoo"]
keep = max(1, round(0.05 * len(gentoo)))
imbalanced = pd.concat([adelie, gentoo.sample(keep, random_state=0)])

X_imb = imbalanced[FEATURES]
y_imb = imbalanced["species"]

print(f"imbalanced test set: {len(imbalanced)} penguins")
print(y_imb.value_counts().to_string())
viz.plot_class_balance(y_imb)
plt.show()

### Read the figure

The bar chart makes the imbalance plain: a tall stack of Adélie beside a sliver of Gentoo. This is the deliberately lopsided world we will now score models in — and it is exactly the shape of data where accuracy turns slippery.

In [ ]:
imb_baseline = DummyClassifier(strategy="most_frequent").fit(X_imb, y_imb)
imb_pred = imb_baseline.predict(X_imb)

print(f"baseline accuracy on the imbalanced set: {accuracy_score(y_imb, imb_pred):.3f}")
n_gentoo = int((y_imb == "Gentoo").sum())
print(f"Gentoo (rare class) correctly found: {int((imb_pred == 'Gentoo').sum())} / {n_gentoo}")

### Read the output

Here is the **accuracy paradox**. The baseline scores **0.95** on the imbalanced set — a number that looks excellent — yet it labels every penguin "Adélie" and so identifies **zero** of the Gentoos. If Gentoos were the rare disease, the rare fraud, the thing we built the model to catch, this "95% accurate" model would catch none of them. Accuracy bought entirely by the majority class is no achievement at all.

In [ ]:
imb_clf_pred = clf.predict(X_imb)

print(f"nearest-centroid accuracy on the imbalanced set: {accuracy_score(y_imb, imb_clf_pred):.3f}")
print(f"Gentoo (rare class) correctly found: {int((imb_clf_pred == 'Gentoo').sum())} / {n_gentoo}")

### Read the output

The nearest-centroid classifier scores 1.00 here and finds both Gentoos. Set the two side by side: 0.95 versus 1.00 — barely five points apart on accuracy — yet one model is useless on the rare class and the other is perfect on it. Accuracy alone almost hid that difference. (Two Gentoos is far too few to *measure* how reliably the model finds them — the point is only that accuracy hid the gap, not that this model is proven on the rare class.) To see it properly, we look not at *how many* errors, but at *which* errors: who gets mistaken for whom.

## The positive class

Often one class is the one we actually care about: the patient who has the disease, the transaction that is fraud, the rare species in a survey. We call it the **positive** class, and the other the **negative** class. Most of the measures in the next notebook are defined around it — "did we catch the positives?" So we make a choice and keep it: for the rest of this evaluation thread, **Gentoo is the positive class**.

## So what

Accuracy is a fine first number — quick, intuitive, and a real improvement over guessing when the classes are balanced. But it collapses everything into one fraction and cannot tell you *which* mistakes you make: missed Gentoos? false alarms? Notebook 07 opens the **confusion matrix**, which lays every kind of error out in the open, and from it builds **precision** and **recall**.

## The words we'll keep using

Added this notebook:

- **Accuracy** — the fraction of examples classified correctly.
- **Baseline** — a trivial reference (here, always predict the majority class); a model must beat it.
- **`DummyClassifier`** — scikit-learn's stand-in for such trivial baselines.
- **Class imbalance** — one class much more frequent than another.
- **Accuracy paradox** — a high accuracy that hides poor performance on a rare class.
- **Positive class** — the class of interest, the one we most want to catch (here, Gentoo).

## Your turn

1. A classifier gets 57 of 60 test examples right. What is its accuracy?
2. In a dataset that is 99% class A and 1% class B, what accuracy does "always predict A" achieve? Why would reporting only that number be misleading?
3. Name a real problem where the class you care about is the rare one, and say what it would cost to miss it.

Notebook 07 gives us the tools to answer "which errors?" head on.

## What you built

You turned an informal "fraction right" into a measured, anchored judgment:

- You defined and computed **accuracy**, by hand and with scikit-learn.
- You compared the classifier to a **majority baseline** and saw it clear the bar by a wide margin.
- You built an imbalanced case and met the **accuracy paradox** — 95% accuracy that finds none of the rare class.
- You named the **positive class** (Gentoo) for the work ahead.

Accuracy is now a tool you can use *and* distrust in the right places — exactly the posture the rest of evaluation needs.

## References

- A. Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd ed., O'Reilly, 2022 — chapter 3 (performance measures; why accuracy is not enough for classifiers).
- G. James, D. Witten, T. Hastie, R. Tibshirani, *An Introduction to Statistical Learning*, 2nd ed., Springer, 2021 — chapter 4. DOI: 10.1007/978-1-0716-1418-1

---
Previous: **05 — Your first classifier: nearest centroid** · Next: **07 — The confusion matrix, precision & recall**